# 02 FinBERT Sentiment

`chunks.parquet` を読み込み、FinBERT (yiyanghkust/finbert-tone) で
各チャンクの positive / negative / neutral 確率を推論する。
filing × section 単位で集約して可視化。

In [ ]:
# Cell 1: imports + FinBERT ロード
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import _helpers
import torch
import pandas as pd
from tqdm.auto import tqdm

device = _helpers.get_device()
tokenizer, model = _helpers.load_finbert(device)
print('device:', device, 'labels:', model.config.id2label)


In [ ]:
# Cell 2: chunks 読み込み
df_chunks = pd.read_parquet(_helpers.CHUNKS_PARQUET)
print('chunks:', len(df_chunks))
df_chunks.head(3)


In [ ]:
# Cell 3: バッチ推論 (pos/neg/neu)
BATCH_SIZE = 32
id2label = model.config.id2label
label_names = [id2label[i] for i in range(len(id2label))]

all_probs = []
texts = df_chunks['text'].tolist()
for start in tqdm(range(0, len(texts), BATCH_SIZE), desc='finbert'):
    batch = texts[start:start+BATCH_SIZE]
    enc = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model(**enc)
    probs = out.logits.softmax(dim=-1).cpu().numpy()
    all_probs.extend(probs.tolist())
print('done. samples:', len(all_probs))


In [ ]:
# Cell 4: sentiments.parquet 保存
import numpy as np
probs_arr = np.array(all_probs)
# 列名を明示的に pos/neg/neu に統一
name2idx = {n.lower(): i for i, n in enumerate(label_names)}
df_sent = df_chunks[['filing_id','ticker','form','section_key','chunk_idx','filing_date']].copy()
df_sent['pos'] = probs_arr[:, name2idx['positive']]
df_sent['neg'] = probs_arr[:, name2idx['negative']]
df_sent['neu'] = probs_arr[:, name2idx['neutral']]
df_sent['label'] = [label_names[i] for i in probs_arr.argmax(axis=1)]
df_sent.to_parquet(_helpers.SENTIMENTS_PARQUET)
print('saved:', _helpers.SENTIMENTS_PARQUET, 'rows:', len(df_sent))
df_sent.head()


In [ ]:
# Cell 5: filing × section 単位で平均センチメント集約
agg = df_sent.groupby(['ticker','form','section_key','filing_id','filing_date'])[['pos','neg','neu']].mean().reset_index()
agg = agg.sort_values(['ticker','section_key','filing_date'])
agg.head(10)


In [ ]:
# Cell 6: AAPL の Risk Factors (Item 1A) の neg スコア推移
import plotly.express as px
aapl_risk = agg[(agg['ticker']=='AAPL') & (agg['section_key']=='item_1a')].copy()
fig = px.line(
    aapl_risk, x='filing_date', y='neg', color='form', markers=True,
    title='AAPL Risk Factors (Item 1A) - FinBERT negative score over time',
)
fig.show()


In [ ]:
# Cell 7: 3 銘柄 × MD&A センチメント比較 (pos - neg)
mda = agg[agg['section_key']=='item_7'].copy()
mda['net_sentiment'] = mda['pos'] - mda['neg']
fig = px.line(
    mda, x='filing_date', y='net_sentiment', color='ticker', markers=True, line_dash='form',
    title='MD&A net sentiment (pos - neg) by ticker/form',
)
fig.show()
